In [1]:
import random

def generate_sudoku(difficulty='orta'):
    """
    Rastgele geçerli bir Sudoku tahtası üretir.
    difficulty: 'kolay' (30 boş), 'orta' (40 boş), 'zor' (52 boş)
    """
    remove_map = {'kolay': 30, 'orta': 40, 'zor': 52}
    cells_to_remove = remove_map.get(difficulty, 40)

    board = [[0]*9 for _ in range(9)]

    def is_valid(board, row, col, num):
        if num in board[row]: return False
        if num in [board[i][col] for i in range(9)]: return False
        box_r, box_c = 3*(row//3), 3*(col//3)
        for i in range(box_r, box_r+3):
            for j in range(box_c, box_c+3):
                if board[i][j] == num: return False
        return True

    def solve(board):
        for i in range(9):
            for j in range(9):
                if board[i][j] == 0:
                    nums = list(range(1, 10))
                    random.shuffle(nums)
                    for num in nums:
                        if is_valid(board, i, j, num):
                            board[i][j] = num
                            if solve(board): return True
                            board[i][j] = 0
                    return False
        return True

    solve(board)
    solution = [row[:] for row in board]

    positions = [(i, j) for i in range(9) for j in range(9)]
    random.shuffle(positions)
    puzzle = [row[:] for row in board]
    for i, j in positions[:cells_to_remove]:
        puzzle[i][j] = 0

    return puzzle, solution

puzzle, solution = generate_sudoku('orta')
print('Sudoku tahtası üretildi!')
print('Boş hücre sayısı:', sum(1 for row in puzzle for val in row if val == 0))

Sudoku tahtası üretildi!
Boş hücre sayısı: 40


In [2]:
from IPython.display import HTML
import json

def create_sudoku_html(puzzle, solution):
    puzzle_json   = json.dumps(puzzle)
    solution_json = json.dumps(solution)

    html = f"""
    <style>
      * {{ box-sizing: border-box; }}
      #sudoku-game {{
        font-family: 'Segoe UI', sans-serif;
        max-width: 540px;
        margin: 16px auto;
        padding: 20px;
        background: #f8f9fa;
        border-radius: 14px;
        box-shadow: 0 4px 24px rgba(0,0,0,0.12);
      }}

      /* ── Üst bilgi çubuğu ── */
      #top-bar {{
        display: flex;
        justify-content: space-between;
        align-items: center;
        margin-bottom: 10px;
        flex-wrap: wrap;
        gap: 6px;
      }}
      h2 {{ color: #2c3e50; margin: 0; font-size: 22px; }}
      #timer-box {{
        font-size: 20px;
        font-weight: 700;
        color: #2c3e50;
        background: #fff;
        border: 2px solid #3498db;
        border-radius: 8px;
        padding: 4px 14px;
        min-width: 90px;
        text-align: center;
      }}
      #timer-box.danger {{ border-color: #e74c3c; color: #e74c3c; animation: pulse 1s infinite; }}
      @keyframes pulse {{ 0%,100%{{opacity:1}} 50%{{opacity:.5}} }}

      /* ── Zorluk seçici ── */
      #difficulty-bar {{
        display: flex;
        justify-content: center;
        gap: 8px;
        margin-bottom: 10px;
      }}
      .diff-btn {{
        padding: 6px 16px;
        border: 2px solid #bdc3c7;
        border-radius: 20px;
        background: white;
        font-size: 13px;
        font-weight: 600;
        cursor: pointer;
        transition: all .15s;
        color: #7f8c8d;
      }}
      .diff-btn:hover {{ border-color: #3498db; color: #3498db; }}
      .diff-btn.active {{ background: #3498db; border-color: #3498db; color: white; }}

      /* ── İpucu sayacı ── */
      #hint-counter {{
        text-align: center;
        color: #7f8c8d;
        font-size: 13px;
        margin-bottom: 10px;
      }}

      /* ── Tahta ── */
      #board {{
        display: grid;
        grid-template-columns: repeat(9, 1fr);
        border: 3px solid #2c3e50;
        border-radius: 4px;
        overflow: hidden;
        margin-bottom: 14px;
      }}
      .cell {{
        width: 100%;
        aspect-ratio: 1;
        border: 1px solid #bdc3c7;
        text-align: center;
        font-size: 18px;
        font-weight: 500;
        cursor: pointer;
        background: white;
        color: #2c3e50;
        display: flex;
        align-items: center;
        justify-content: center;
        transition: background .15s;
        user-select: none;
      }}
      .cell.given    {{ color:#1a252f; font-weight:700; background:#ecf0f1; cursor:default; }}
      .cell.selected {{ background:#3498db22 !important; outline:2px solid #3498db; outline-offset:-2px; }}
      .cell.highlight{{ background:#3498db11 !important; }}
      .cell.same-num {{ background:#f39c1233 !important; }}
      .cell.error    {{ background:#e74c3c22 !important; color:#c0392b; }}
      .cell.correct  {{ background:#27ae6022 !important; }}
      .cell.hinted   {{ color:#8e44ad; font-weight:700; background:#9b59b622 !important; }}
      .cell:nth-child(3n)                        {{ border-right:2.5px solid #2c3e50; }}
      .cell:nth-child(n+19):nth-child(-n+27),
      .cell:nth-child(n+46):nth-child(-n+54)     {{ border-bottom:2.5px solid #2c3e50; }}

      /* ── Numpad ── */
      #numpad {{
        display: grid;
        grid-template-columns: repeat(9, 1fr);
        gap: 5px;
        margin-bottom: 12px;
      }}
      .numkey {{
        padding: 10px 0;
        border: none;
        border-radius: 6px;
        background: #3498db;
        color: white;
        font-size: 16px;
        font-weight: 600;
        cursor: pointer;
        transition: background .15s, transform .1s;
      }}
      .numkey:hover {{ background:#2980b9; transform:translateY(-1px); }}

      /* ── Kontrol butonları ── */
      #controls {{ display:flex; gap:8px; justify-content:center; flex-wrap:wrap; }}
      .btn {{
        padding: 9px 16px;
        border: none;
        border-radius: 8px;
        font-size: 13px;
        font-weight: 600;
        cursor: pointer;
        transition: background .15s, opacity .15s;
      }}
      #btn-clear {{ background:#e67e22; color:white; }}
      #btn-clear:hover {{ background:#d35400; }}
      #btn-check {{ background:#27ae60; color:white; }}
      #btn-check:hover {{ background:#1e8449; }}
      #btn-hint  {{ background:#8e44ad; color:white; }}
      #btn-hint:hover  {{ background:#7d3c98; }}
      #btn-hint:disabled {{ opacity:.45; cursor:not-allowed; }}
      #btn-new   {{ background:#2c3e50; color:white; }}
      #btn-new:hover   {{ background:#1a252f; }}
      #btn-scores{{ background:#e74c3c; color:white; }}
      #btn-scores:hover{{ background:#c0392b; }}

      /* ── Durum mesajı ── */
      #status {{
        text-align:center;
        margin-top:10px;
        font-size:14px;
        font-weight:500;
        color:#7f8c8d;
        min-height:20px;
      }}

      /* ── Skor tablosu modal ── */
      #score-modal {{
        display: none;
        position: fixed;
        inset: 0;
        background: rgba(0,0,0,.5);
        z-index: 999;
        align-items: center;
        justify-content: center;
      }}
      #score-modal.open {{ display:flex; }}
      #score-box {{
        background: white;
        border-radius: 14px;
        padding: 28px 32px;
        width: 340px;
        max-width: 92vw;
        box-shadow: 0 8px 40px rgba(0,0,0,.25);
      }}
      #score-box h3 {{
        margin: 0 0 16px;
        color: #2c3e50;
        font-size: 18px;
        text-align: center;
      }}
      .score-tabs {{
        display: flex;
        gap: 6px;
        margin-bottom: 14px;
        justify-content: center;
      }}
      .score-tab {{
        padding: 5px 14px;
        border: 2px solid #bdc3c7;
        border-radius: 16px;
        font-size: 12px;
        font-weight: 600;
        cursor: pointer;
        color: #7f8c8d;
        background: white;
        transition: all .15s;
      }}
      .score-tab.active {{ background:#3498db; border-color:#3498db; color:white; }}
      #score-list {{
        list-style: none;
        margin: 0;
        padding: 0;
        max-height: 260px;
        overflow-y: auto;
      }}
      #score-list li {{
        display: flex;
        justify-content: space-between;
        padding: 8px 10px;
        border-radius: 6px;
        font-size: 14px;
        color: #2c3e50;
      }}
      #score-list li:nth-child(odd)  {{ background:#f8f9fa; }}
      #score-list li .rank {{ font-weight:700; color:#3498db; min-width:24px; }}
      #score-list li .time {{ font-weight:600; }}
      #score-list li .meta {{ color:#95a5a6; font-size:12px; }}
      #score-list .empty  {{ justify-content:center; color:#bdc3c7; font-style:italic; }}
      #score-footer {{
        display: flex;
        justify-content: space-between;
        margin-top: 14px;
      }}
      .modal-btn {{
        padding: 8px 18px;
        border: none;
        border-radius: 7px;
        font-size: 13px;
        font-weight: 600;
        cursor: pointer;
      }}
      #btn-clear-scores {{ background:#e74c3c; color:white; }}
      #btn-close-scores  {{ background:#2c3e50;  color:white; }}

      /* ── Tamamlama banner ── */
      #win-banner {{
        display: none;
        text-align: center;
        padding: 14px;
        background: linear-gradient(135deg,#27ae60,#2ecc71);
        color: white;
        border-radius: 10px;
        margin-top: 12px;
        font-size: 16px;
        font-weight: 700;
        animation: fadeIn .4s ease;
      }}
      @keyframes fadeIn {{ from{{opacity:0;transform:translateY(-8px)}} to{{opacity:1;transform:none}} }}
    </style>

    <!-- Modal: Skor Tablosu -->
    <div id="score-modal">
      <div id="score-box">
        <h3>🏆 Skor Tablosu</h3>
        <div class="score-tabs">
          <button class="score-tab active" onclick="showScores('kolay')">Kolay</button>
          <button class="score-tab"        onclick="showScores('orta')">Orta</button>
          <button class="score-tab"        onclick="showScores('zor')">Zor</button>
        </div>
        <ul id="score-list"></ul>
        <div id="score-footer">
          <button class="modal-btn" id="btn-clear-scores" onclick="clearScores()">Sıfırla</button>
          <button class="modal-btn" id="btn-close-scores" onclick="closeScoreModal()">Kapat</button>
        </div>
      </div>
    </div>

    <!-- Ana oyun alanı -->
    <div id="sudoku-game">

      <div id="top-bar">
        <h2>🎯 Sudoku</h2>
        <div id="timer-box">00:00</div>
      </div>

      <div id="difficulty-bar">
        <button class="diff-btn" onclick="changeDifficulty('kolay')">Kolay</button>
        <button class="diff-btn active" onclick="changeDifficulty('orta')">Orta</button>
        <button class="diff-btn" onclick="changeDifficulty('zor')">Zor</button>
      </div>

      <div id="hint-counter">İpucu hakkı: <span id="hints-left">3</span> / 3</div>

      <div id="board"></div>

      <div id="numpad">
        {''.join(f'<button class="numkey" onclick="enterNum({i})">{i}</button>' for i in range(1,10))}
      </div>

      <div id="controls">
        <button class="btn" id="btn-clear"  onclick="clearCell()">⌫ Sil</button>
        <button class="btn" id="btn-check"  onclick="checkBoard()">✔ Kontrol</button>
        <button class="btn" id="btn-hint"   onclick="useHint()">💡 İpucu</button>
        <button class="btn" id="btn-new"    onclick="newGame()">🔄 Yeni</button>
        <button class="btn" id="btn-scores" onclick="openScoreModal()">🏆 Skorlar</button>
      </div>

      <div id="status">Bir hücre seç, ardından numpad veya klavyeden sayı gir.</div>
      <div id="win-banner"></div>
    </div>

    <script>
    /* ════════════════════════════════════════
       SUDOKU — Gelişmiş sürüm
       ✔ Zamanlayıcı
       ✔ Zorluk seviyesi
       ✔ İpucu butonu
       ✔ Klavye desteği (1-9, Delete/Backspace, ok tuşları)
       ✔ Skor tablosu (localStorage)
       ✔ Satır/sütun/kutu vurgusu
       ════════════════════════════════════════ */

    // ── Tahta verileri (Python tarafından enjekte edilir) ──
    let puzzle   = {puzzle_json};
    let solution = {solution_json};

    // ── Oyun durumu ──
    let selected    = null;
    let userBoard   = puzzle.map(r => [...r]);
    let hintedCells = {{}};
    let hintsLeft   = 3;
    let currentDiff = 'orta';
    let gameOver    = false;
    let scoreTabDiff= 'kolay';

    // ── Zamanlayıcı ──
    let seconds = 0;
    let timerID = null;

    function startTimer() {{
      stopTimer();
      seconds = 0;
      updateTimerDisplay();
      timerID = setInterval(() => {{
        seconds++;
        updateTimerDisplay();
      }}, 1000);
    }}
    function stopTimer()  {{ clearInterval(timerID); timerID = null; }}
    function updateTimerDisplay() {{
      const m = String(Math.floor(seconds/60)).padStart(2,'0');
      const s = String(seconds % 60).padStart(2,'0');
      const el = document.getElementById('timer-box');
      el.textContent = m+':'+s;
      el.classList.toggle('danger', seconds >= 600);  // 10 dk sonra kırmızı
    }}
    function fmtTime(sec) {{
      const m = String(Math.floor(sec/60)).padStart(2,'0');
      const s = String(sec%60).padStart(2,'0');
      return m+':'+s;
    }}

    // ── Zorluk değiştir ──
    function changeDifficulty(diff) {{
      document.querySelectorAll('.diff-btn').forEach(b => b.classList.remove('active'));
      const labels = {{'kolay':0,'orta':1,'zor':2}};
      document.querySelectorAll('.diff-btn')[labels[diff]].classList.add('active');
      currentDiff = diff;
      newGame();
    }}

    // ── Yeni oyun — Python ürettiği tahtayı kullanır
    //    (gerçek yeni üretim için kernel'i yeniden çalıştırın;
    //     ama şu an çalışır tahtayı sıfırlar)
    function newGame() {{
      userBoard   = puzzle.map(r => [...r]);
      hintedCells = {{}};
      hintsLeft   = 3;
      selected    = null;
      gameOver    = false;
      document.getElementById('hints-left').textContent = 3;
      document.getElementById('btn-hint').disabled = false;
      document.getElementById('win-banner').style.display = 'none';
      setStatus('Yeni oyun başladı! Bir hücre seç.');
      startTimer();
      renderBoard();
    }}

    // ── Tahta render ──
    function renderBoard() {{
      const boardEl = document.getElementById('board');
      boardEl.innerHTML = '';
      const [sr, sc] = selected ? selected : [-1,-1];
      const selVal    = (sr>=0 && userBoard[sr][sc] !== 0) ? userBoard[sr][sc] : -1;

      for (let i=0; i<9; i++) {{
        for (let j=0; j<9; j++) {{
          const cell     = document.createElement('div');
          const isGiven  = puzzle[i][j] !== 0;
          const isHinted = hintedCells[i+','+j];
          cell.className = 'cell';
          if (isGiven)  cell.classList.add('given');
          if (isHinted) cell.classList.add('hinted');

          // Seçili hücre
          if (i===sr && j===sc) {{
            cell.classList.add('selected');
          }} else if (sr>=0) {{
            const sameBox = Math.floor(i/3)===Math.floor(sr/3) && Math.floor(j/3)===Math.floor(sc/3);
            if (i===sr || j===sc || sameBox) cell.classList.add('highlight');
          }}

          // Aynı rakam vurgulama
          if (selVal>0 && userBoard[i][j]===selVal && !(i===sr && j===sc))
            cell.classList.add('same-num');

          cell.textContent = userBoard[i][j] !== 0 ? userBoard[i][j] : '';

          if (!isGiven) {{
            cell.onclick = () => {{
              if (gameOver) return;
              selected = [i,j];
              renderBoard();
            }};
          }}
          boardEl.appendChild(cell);
        }}
      }}
    }}

    // ── Sayı girişi ──
    function enterNum(n) {{
      if (!selected || gameOver) {{ setStatus('Önce bir hücre seç!'); return; }}
      const [r,c] = selected;
      if (puzzle[r][c] !== 0 || hintedCells[r+','+c]) return;
      userBoard[r][c] = n;
      renderBoard();
      setStatus('');
      checkWin();
    }}

    // ── Hücre sil ──
    function clearCell() {{
      if (!selected || gameOver) return;
      const [r,c] = selected;
      if (puzzle[r][c] !== 0 || hintedCells[r+','+c]) {{ setStatus('Bu hücre silinemez.'); return; }}
      userBoard[r][c] = 0;
      renderBoard();
    }}

    // ── İpucu ──
    function useHint() {{
      if (hintsLeft <= 0) {{ setStatus('İpucu hakkın kalmadı!'); return; }}
      if (gameOver) return;
      let tr=-1, tc=-1;
      if (selected) {{
        const [r,c]=selected;
        if (puzzle[r][c]===0 && !hintedCells[r+','+c] && userBoard[r][c]!==solution[r][c]) {{
          tr=r; tc=c;
        }} else if (puzzle[r][c]!==0 || hintedCells[r+','+c]) {{
          setStatus('Bu hücre zaten dolu.'); return;
        }}
      }}
      if (tr===-1) {{
        const empties=[];
        for(let i=0;i<9;i++) for(let j=0;j<9;j++)
          if(puzzle[i][j]===0 && !hintedCells[i+','+j] && userBoard[i][j]!==solution[i][j])
            empties.push([i,j]);
        if(!empties.length) {{ setStatus('Doldurulacak hücre kalmadı!'); return; }}
        [tr,tc]=empties[Math.floor(Math.random()*empties.length)];
      }}
      userBoard[tr][tc]=solution[tr][tc];
      hintedCells[tr+','+tc]=true;
      selected=[tr,tc];
      hintsLeft--;
      document.getElementById('hints-left').textContent=hintsLeft;
      if(hintsLeft===0) document.getElementById('btn-hint').disabled=true;
      renderBoard();
      setStatus(`💡 (${{tr+1}},${{tc+1}}) hücresine ipucu. Kalan: ${{hintsLeft}}`);
      checkWin();
    }}

    // ── Kontrol ──
    function checkBoard() {{
      if (gameOver) return;
      let errors=0, filled=0;
      const freeCells=document.querySelectorAll('.cell:not(.given):not(.hinted)');
      let idx=0;
      for(let i=0;i<9;i++) {{
        for(let j=0;j<9;j++) {{
          if(puzzle[i][j]===0 && !hintedCells[i+','+j]) {{
            freeCells[idx].classList.remove('error','correct');
            if(userBoard[i][j]!==0) {{
              filled++;
              if(userBoard[i][j]!==solution[i][j]) {{ freeCells[idx].classList.add('error'); errors++; }}
              else freeCells[idx].classList.add('correct');
            }}
            idx++;
          }}
        }}
      }}
      const totalFree=document.querySelectorAll('.cell:not(.given)').length;
      const hintN=Object.keys(hintedCells).length;
      const allFilled=filled+hintN===totalFree;
      if(errors===0 && allFilled) finishGame();
      else if(errors>0) setStatus(`❌ ${{errors}} hatalı hücre var.`);
      else setStatus('✅ Şimdiye kadar hepsi doğru! Devam et.');
    }}

    // ── Otomatik kazanma kontrolü ──
    function checkWin() {{
      for(let i=0;i<9;i++) for(let j=0;j<9;j++)
        if(userBoard[i][j]!==solution[i][j]) return;
      finishGame();
    }}

    // ── Oyun bitti ──
    function finishGame() {{
      stopTimer();
      gameOver=true;
      const hintUsed=3-hintsLeft;
      saveScore(currentDiff, seconds, hintUsed);
      const banner=document.getElementById('win-banner');
      banner.style.display='block';
      banner.textContent=`🎉 Tebrikler! Süre: ${{fmtTime(seconds)}} | Kullanılan ipucu: ${{hintUsed}}`;
      setStatus('');
    }}

    function setStatus(msg) {{ document.getElementById('status').textContent=msg; }}

    // ════════════════════════════════════════
    // SKOR TABLOSU
    // ════════════════════════════════════════
    const SCORE_KEY='sudoku_scores_v1';

    function loadScores() {{
      try {{ return JSON.parse(localStorage.getItem(SCORE_KEY)||'{{}}'); }}
      catch(e) {{ return {{}}; }}
    }}
    function saveScores(data) {{
      try {{ localStorage.setItem(SCORE_KEY, JSON.stringify(data)); }} catch(e){{}}
    }}
    function saveScore(diff, sec, hints) {{
      const all=loadScores();
      if(!all[diff]) all[diff]=[];
      const date=new Date().toLocaleDateString('tr-TR');
      all[diff].push({{sec,hints,date}});
      all[diff].sort((a,b)=>a.sec-b.sec);
      all[diff]=all[diff].slice(0,10);  // en iyi 10
      saveScores(all);
    }}
    function clearScores() {{
      if(confirm('Tüm skorlar silinsin mi?')) {{
        localStorage.removeItem(SCORE_KEY);
        showScores(scoreTabDiff);
      }}
    }}
    function openScoreModal() {{
      showScores('kolay');
      document.getElementById('score-modal').classList.add('open');
    }}
    function closeScoreModal() {{
      document.getElementById('score-modal').classList.remove('open');
    }}
    function showScores(diff) {{
      scoreTabDiff=diff;
      document.querySelectorAll('.score-tab').forEach((t,i)=>{{
        t.classList.toggle('active',['kolay','orta','zor'][i]===diff);
      }});
      const all=loadScores();
      const list=document.getElementById('score-list');
      list.innerHTML='';
      const rows=(all[diff]||[]);
      if(!rows.length) {{
        list.innerHTML='<li class="empty">Henüz kayıt yok</li>';
        return;
      }}
      rows.forEach((r,i)=>{{
        const medals=['🥇','🥈','🥉'];
        const li=document.createElement('li');
        li.innerHTML=`<span class="rank">${{medals[i]||'#'+(i+1)}}</span>
          <span class="time">${{fmtTime(r.sec)}}</span>
          <span class="meta">💡${{r.hints}} | ${{r.date}}</span>`;
        list.appendChild(li);
      }});
    }}

    // ════════════════════════════════════════
    // KLAVYe DESTEĞİ
    // ════════════════════════════════════════
    document.addEventListener('keydown', (e) => {{
      if (gameOver) return;

      // Rakam tuşları 1-9
      if (e.key>='1' && e.key<='9') {{
        enterNum(parseInt(e.key)); return;
      }}

      // Silme
      if (e.key==='Backspace' || e.key==='Delete' || e.key==='0') {{
        clearCell(); return;
      }}

      // Ok tuşları — seçili hücreyi hareket ettirir
      const moves={{'ArrowUp':[-1,0],'ArrowDown':[1,0],'ArrowLeft':[0,-1],'ArrowRight':[0,1]}};
      if (moves[e.key]) {{
        e.preventDefault();
        if (!selected) {{ selected=[0,0]; renderBoard(); return; }}
        let [r,c]=selected;
        const [dr,dc]=moves[e.key];
        r=Math.max(0,Math.min(8,r+dr));
        c=Math.max(0,Math.min(8,c+dc));
        selected=[r,c];
        // Verilen hücreleri atla
        renderBoard();
      }}

      // H → ipucu, C → kontrol, N → yeni
      if (e.key==='h'||e.key==='H') useHint();
      if (e.key==='c'||e.key==='C') checkBoard();
      if (e.key==='n'||e.key==='N') newGame();
    }});

    // Modal dışına tıklayınca kapat
    document.getElementById('score-modal').addEventListener('click', (e)=>{{
      if(e.target===document.getElementById('score-modal')) closeScoreModal();
    }});

    // ── Başlat ──
    startTimer();
    renderBoard();
    </script>
    """
    return HTML(html)

create_sudoku_html(puzzle, solution)